# Bound curves for MMLS and beta-VAE on dSprites (Figure 4)

Requires existing artifacts from the [sweep runner](../../experiments_and_analysis/scripts/sweep.py)
using [figure4.yaml](../../experiments_and_analysis/scripts/configs/paper/figure4.yaml);
see the [reproduction instructions](../../docs/source/guide/reproducibility.rst).
Set `RESULTS_ROOT` below; if several matching sweeps exist, set `SWEEP_DIRECTORY` explicitly.

The [quick preset](../../experiments_and_analysis/scripts/configs/paper/figure4_quick.yaml) is also supported.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'pyproject.toml').is_file() and (path / 'microscope').is_dir()
)
sys.path.insert(0, str(REPO_ROOT))

from experiments_and_analysis.outputs import read_json
from experiments_and_analysis.plotting.render import bound_data, render

In [ ]:
RESULTS_ROOT = REPO_ROOT.parent / 'outputs' / 'NEW_PAPER_REPRODUCTION'
SWEEP_DIRECTORY = None  # Or Path('/path/to/experiment-configurations/sweep__figure_4__...').

if SWEEP_DIRECTORY is None:
    candidates = sorted(
        path.parent for path in
        (RESULTS_ROOT / 'experiment-configurations').glob('sweep__figure_4__*/results.json')
    )
    if len(candidates) != 1:
        raise ValueError(
            f'Expected one completed Figure 4 sweep under {RESULTS_ROOT}; found {len(candidates)}. '
            'Set SWEEP_DIRECTORY to the desired sweep directory.'
        )
    SWEEP_DIRECTORY = candidates[0]

## Load completed reconstruction-error analyses

In [ ]:
config = read_json(SWEEP_DIRECTORY / 'config.json')
result = read_json(SWEEP_DIRECTORY / 'results.json')
request = next(output for output in config['outputs'] if output['filename'] == 'figure_4')
records = [record for experiment in result['results'] for record in experiment['analyses']]
curves, summaries = bound_data(records, request['panels'])
pd.DataFrame(summaries).T

## Plot and export the bound curves

In [ ]:
# Reuse or render the plot from completed analyses; no models are trained here.
plot_directory = render(records, request, RESULTS_ROOT)
display(Image(filename=str(plot_directory / 'figure_4.png')))

# The runner also saves curves.npz and regression.json beside its aggregate manifest.
plot_directory